# ILS Loss Estimation Model - Calibration

**Objective**: Estimate the distribution of industry-wide insured losses given an earthquake magnitude and geographic zone.

**Context**: ~18.8% of outstanding cat bonds use Industry Loss triggers (Artemis, 2025). When a seismic event occurs, the key question is: *what is the probability that industry-wide insured losses breach the attachment point?*

**Approach**:
1. Log-linear regression: `log10(loss) ~ magnitude + log10(penetration)`
2. GEV distribution on residuals - selected by AIC over Normal
3. Combined: full predictive distribution of insured losses

**Data source**: EM-DAT (2000–2024), 76 earthquakes with magnitude AND adjusted insured damage.

**Limitations** (acknowledged upfront):
- R² ~0.30: magnitude and penetration explain ~30% of variance - residual uncertainty is explicitly modelled
- 76 observations: limited data in the extreme tail - GPD on exceedances would be preferred with more data
- Industry loss triggers only (~18.8% of cat bond market, Artemis 2025)
- In production: replace with RMS/AIR outputs and PCS/PERILS data

---

## 1. Data Loading and Preparation

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import genextreme, norm, kstest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import json

EMDAT_FILE = "data/public_emdat.xlsx"
df_raw = pd.read_excel(EMDAT_FILE)

COL_INSURED_ADJ = "Insured Damage, Adjusted ('000 US$)"

# Filter: earthquakes with magnitude AND inflation-adjusted insured damage
df = df_raw[
    (df_raw["Disaster Type"] == "Earthquake") &
    df_raw["Magnitude"].notna() &
    df_raw[COL_INSURED_ADJ].notna()
].copy()

# Convert to $bn - inflation-adjusted figures used throughout
df["loss_bn"] = df[COL_INSURED_ADJ] / 1_000_000
df["log_loss"] = np.log10(df["loss_bn"])

print(f"Dataset: {len(df)} earthquakes | Period: {df['Start Year'].min()}–{df['Start Year'].max()}")
print(f"Magnitude: {df['Magnitude'].min():.1f} – {df['Magnitude'].max():.1f}")
print(f"Insured loss: ${df['loss_bn'].min():.4f}bn – ${df['loss_bn'].max():.1f}bn")

Dataset: 76 earthquakes | Period: 2000–2024
Magnitude: 4.6 – 9.1
Insured loss: $0.0001bn – $53.7bn


## 2. Exploratory Analysis

Two key observations:
- Insured losses are **heavy-tailed**: mean >> median - a few extreme events dominate
- There is a **positive relationship** between magnitude and log(loss), but with large scatter driven by geographic differences in insurance penetration

In [9]:
mean_loss = df["loss_bn"].mean()
median_loss = df["loss_bn"].median()
print(f"Mean loss   : ${mean_loss:.3f}bn")
print(f"Median loss : ${median_loss:.3f}bn")
print(f"Ratio       : {mean_loss/median_loss:.0f}x")
print("=> Heavy-tailed distribution: a few extreme events dominate total losses")

Mean loss   : $1.712bn
Median loss : $0.167bn
Ratio       : 10x
=> Heavy-tailed distribution: a few extreme events dominate total losses


In [10]:
# Raw vs log distribution - justifies log-scale regression
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Raw scale - extreme right skew", "Log scale - approximately symmetric")
)
fig.add_trace(go.Histogram(x=df["loss_bn"], nbinsx=25,
    marker_color="steelblue", name="Raw"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["log_loss"], nbinsx=20,
    marker_color="steelblue", name="Log"), row=1, col=2)
fig.update_xaxes(title_text="Insured Loss ($bn)", row=1, col=1)
fig.update_xaxes(title_text="Log10(Insured Loss $bn)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(
    title="Distribution of Earthquake Insured Losses - EM-DAT 2000-2024",
    showlegend=False, height=400
)
fig.show(renderer="iframe")
print("=> Log transformation linearises the multiplicative relationship between magnitude and losses")

=> Log transformation linearises the multiplicative relationship between magnitude and losses


In [11]:
# Magnitude vs log(loss) - key relationship
fig = px.scatter(
    df, x="Magnitude", y="log_loss",
    color="Country",
    hover_name="Country",
    hover_data={"Start Year": True, "loss_bn": ":.3f", "Magnitude": True},
    trendline="ols", trendline_scope="overall",
    title="Magnitude vs Log10(Insured Loss) - Earthquakes 2000-2024",
    labels={"Magnitude": "Magnitude", "log_loss": "Log10(Insured Loss $bn)"}
)
fig.show(renderer="iframe")
print("Two observations:")
print("1. Positive trend: higher magnitude -> higher insured losses")
print("2. Large scatter: at M7.0, losses range from $0.001bn to $1bn+")
print("   -> Geographic effect (Japan vs Indonesia) explains much of the scatter")
print("   -> This motivates adding insurance penetration as a second variable")

Two observations:
1. Positive trend: higher magnitude -> higher insured losses
2. Large scatter: at M7.0, losses range from $0.001bn to $1bn+
   -> Geographic effect (Japan vs Indonesia) explains much of the scatter
   -> This motivates adding insurance penetration as a second variable


## 3. Insurance Penetration - Geographic Adjustment

A M7.0 earthquake in Japan generates far more insured losses than the same event in Indonesia - not because the physical destruction differs, but because Japan insures ~28% of economic losses vs ~3% in Indonesia.

**Penetration = insured damage / total damage (median by country, from EM-DAT)**

Countries with fewer than 3 observations get a conservative 5% default.
Note: EM-DAT is biased towards high-penetration countries - countries missing from the dataset are likely low-penetration markets, making 5% an appropriate conservative fallback.

In [12]:
# Calculate insurance penetration directly from EM-DAT
# penetration = insured damage / total damage (median by country)
COL_TOTAL_ADJ = "Total Damage, Adjusted ('000 US$)"

df_pen = df_raw[
    (df_raw["Disaster Type"] == "Earthquake") &
    df_raw[COL_INSURED_ADJ].notna() &
    df_raw[COL_TOTAL_ADJ].notna()
].copy()

df_pen["penetration_calculated"] = (
    df_pen[COL_INSURED_ADJ] / df_pen[COL_TOTAL_ADJ]
)

# Only trust countries with >= 3 observations
reliable_countries = (df_pen.groupby("Country")["penetration_calculated"]
                      .count()[lambda x: x >= 3].index)

penetration_reliable = (df_pen[df_pen["Country"].isin(reliable_countries)]
                        .groupby("Country")["penetration_calculated"].median())

# Show observation count and penetration for reliable countries
obs_count = df_pen.groupby("Country")["penetration_calculated"].count()
print("Reliable countries (>= 3 observations):")
summary = pd.DataFrame({
    "N obs": obs_count[reliable_countries],
    "Penetration": penetration_reliable
}).sort_values("N obs", ascending=False)
print(summary.round(3))

# Map to main dataset - 5% fallback for countries with insufficient data
df["penetration"] = df["Country"].map(penetration_reliable).fillna(0.05)
df["log_penetration"] = np.log10(df["penetration"])

print("\nFinal penetration by country:")
print(df.groupby("Country")["penetration"].first()
      .sort_values(ascending=False).round(3))

Reliable countries (>= 3 observations):
                            N obs  Penetration
Country                                       
Japan                          16        0.277
Indonesia                       6        0.033
China                           5        0.004
Mexico                          5        0.333
New Zealand                     5        0.667
Italy                           4        0.093
United States of America        4        0.232
Taiwan (Province of China)      3        0.400
Türkiye                         3        0.016

Final penetration by country:
Country
New Zealand                                             0.667
Taiwan (Province of China)                              0.400
Mexico                                                  0.333
Japan                                                   0.277
United States of America                                0.232
Italy                                                   0.093
Greece                          

## 4. Log-Linear Regression Model

**Model**: `log10(loss) = β0 + β1 × magnitude + β2 × log10(penetration) + ε`

**Why log-linear?**
- Losses are multiplicative: each unit of magnitude multiplies losses by a constant factor
- Magnitude is already a logarithmic scale - adding log10(magnitude) would mean log of a log
- Penetration effect is multiplicative: doubling penetration always doubles insured losses

Consistent with the USGS PAGER methodology for rapid earthquake loss estimation.

In [13]:
X = df[["Magnitude", "log_penetration"]].values
y = df["log_loss"].values

model = LinearRegression()
model.fit(X, y)
r2 = r2_score(y, model.predict(X))

print("REGRESSION MODEL")
print("=" * 50)
print(f"  log10(loss) = {model.intercept_:.3f}")
print(f"             + {model.coef_[0]:.3f} x magnitude")
print(f"             + {model.coef_[1]:.3f} x log10(penetration)")
print(f"  R2 = {r2:.3f}")
print()
print("Interpretation:")
print(f"  Each +1 magnitude -> losses x {10**model.coef_[0]:.2f} (beta1={model.coef_[0]:.3f})")
print(f"  Doubling penetration -> losses x {2**model.coef_[1]:.2f} (beta2={model.coef_[1]:.3f})")
print(f"  R2 = {r2:.3f} - intentionally low: magnitude and penetration capture the trend,")
print(f"  residual uncertainty is explicitly modelled in the next section")
print()
print("Note: In ILS applications, a distribution is more useful than a point estimate.")
print("The team needs to know p50, p90, p95 - not just the mean prediction.")

# Store predictions and residuals
df["pred"] = model.predict(X)
df["residual"] = df["log_loss"] - df["pred"]

REGRESSION MODEL
  log10(loss) = -1.895
             + 0.280 x magnitude
             + 0.801 x log10(penetration)
  R2 = 0.301

Interpretation:
  Each +1 magnitude -> losses x 1.91 (beta1=0.280)
  Doubling penetration -> losses x 1.74 (beta2=0.801)
  R2 = 0.301 - intentionally low: magnitude and penetration capture the trend,
  residual uncertainty is explicitly modelled in the next section

Note: In ILS applications, a distribution is more useful than a point estimate.
The team needs to know p50, p90, p95 - not just the mean prediction.


## 5. Residual Analysis - Distribution Selection

The residuals represent the part of insured losses not explained by magnitude and penetration.
We test two candidate distributions:
- **Normal**: standard assumption, symmetric tails
- **GEV**: allows heavy tails (xi > 0), theoretically justified for extreme value modelling

Selection criteria: AIC (lower = better) and KS test (p > 0.05 = not rejected).

Note: with more data, GPD on exceedances (Peaks Over Threshold) would be preferred - the same methodology as the academic EVT project.

In [14]:
residuals = df["residual"].values

# Fit Normal
mu_n, sigma_n = norm.fit(residuals)
ll_n = np.sum(norm.logpdf(residuals, mu_n, sigma_n))
aic_n = -2*ll_n + 2*2  # k=2 parameters
ks_n = kstest(residuals, lambda x: norm.cdf(x, mu_n, sigma_n))

# Fit GEV
xi_g, mu_g, sigma_g = genextreme.fit(residuals)
ll_g = np.sum(genextreme.logpdf(residuals, xi_g, mu_g, sigma_g))
aic_g = -2*ll_g + 2*3  # k=3 parameters
ks_g = kstest(residuals, lambda x: genextreme.cdf(x, xi_g, mu_g, sigma_g))

print("DISTRIBUTION SELECTION - Regression Residuals")
print("=" * 60)
print(f"{'Model':<10} {'AIC':>8} {'KS stat':>10} {'KS p-value':>12} {'Rejected':>10}")
print("-" * 60)
print(f"{'Normal':<10} {aic_n:>8.2f} {ks_n.statistic:>10.4f} {ks_n.pvalue:>12.4f} {'No' if ks_n.pvalue > 0.05 else 'YES':>10}")
print(f"{'GEV':<10} {aic_g:>8.2f} {ks_g.statistic:>10.4f} {ks_g.pvalue:>12.4f} {'No' if ks_g.pvalue > 0.05 else 'YES':>10}")
print()

# Automatic selection based on AIC
winner = "GEV" if aic_g < aic_n else "Normal"
print(f"Selected: {winner} (lower AIC, both not rejected by KS)")
print(f"GEV parameters: xi={xi_g:.3f}, mu={mu_g:.3f}, sigma={sigma_g:.3f}")
print()
print(f"xi = {xi_g:.3f} > 0 -> Frechet domain -> heavy-tailed residuals")
print("This means extreme losses are more probable than Normal would suggest")
print("Consistent with academic EVT project where Series X also had xi > 0")
print()
print("Limitation: with only 76 obs, ~19 exceedances above 75th percentile")
print("GPD on exceedances would be more rigorous with more data (see EVT project)")

DISTRIBUTION SELECTION - Regression Residuals
Model           AIC    KS stat   KS p-value   Rejected
------------------------------------------------------------
Normal       195.65     0.0733       0.7808         No
GEV          193.05     0.0595       0.9363         No

Selected: GEV (lower AIC, both not rejected by KS)
GEV parameters: xi=0.458, mu=-0.227, sigma=0.921

xi = 0.458 > 0 -> Frechet domain -> heavy-tailed residuals
This means extreme losses are more probable than Normal would suggest
Consistent with academic EVT project where Series X also had xi > 0

Limitation: with only 76 obs, ~19 exceedances above 75th percentile
GPD on exceedances would be more rigorous with more data (see EVT project)


In [15]:
# Q-Q plots: Normal vs GEV on residuals
sorted_res = np.sort(residuals)
n = len(sorted_res)
probs = np.arange(1, n+1) / (n+1)  # Weibull plotting positions

norm_q = norm.ppf(probs, mu_n, sigma_n)
gev_q = genextreme.ppf(probs, xi_g, mu_g, sigma_g)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Q-Q Plot vs Normal (not selected)", "Q-Q Plot vs GEV (selected)")
)

# Normal Q-Q
fig.add_trace(go.Scatter(
    x=norm_q, y=sorted_res, mode="markers",
    marker=dict(color="steelblue", size=7), name="vs Normal"
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[norm_q.min(), norm_q.max()], y=[norm_q.min(), norm_q.max()],
    mode="lines", line=dict(color="red", dash="dash"), name="Perfect fit", showlegend=False
), row=1, col=1)

# GEV Q-Q
fig.add_trace(go.Scatter(
    x=gev_q, y=sorted_res, mode="markers",
    marker=dict(color="darkgreen", size=7), name="vs GEV"
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[gev_q.min(), gev_q.max()], y=[gev_q.min(), gev_q.max()],
    mode="lines", line=dict(color="red", dash="dash"), name="Perfect fit", showlegend=False
), row=1, col=2)

fig.update_xaxes(title_text="Theoretical Normal quantiles", row=1, col=1)
fig.update_xaxes(title_text="Theoretical GEV quantiles", row=1, col=2)
fig.update_yaxes(title_text="Empirical residuals", row=1, col=1)
fig.update_layout(
    title="Q-Q Plots - Regression Residuals: Normal vs GEV",
    height=450
)
fig.show(renderer="iframe")

print("GEV provides better fit in both tails - particularly the upper tail")
print("Upper tail fit is critical for ILS: determines probability of breaching attachment points")

GEV provides better fit in both tails - particularly the upper tail
Upper tail fit is critical for ILS: determines probability of breaching attachment points


## 6. Predictive Loss Distribution

Combining regression (central estimate) + GEV residuals (uncertainty), we obtain a full predictive distribution of insured losses for any earthquake.

**Model**: `log10(loss) ~ GEV(mu_pred + mu_gev, sigma_gev, xi_gev)`

where `mu_pred = beta0 + beta1 x magnitude + beta2 x log10(penetration)` is the regression central estimate.

In [17]:
def predict_loss_distribution(magnitude, penetration,
                               model, xi_g, mu_g, sigma_g,
                               percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]):
    """
    Predict the full distribution of industry-wide insured losses.

    Model: log10(loss) ~ GEV(mu_pred + mu_g, sigma_g, xi_g)
    where mu_pred = regression central estimate.

    Parameters
    ----------
    magnitude : float - earthquake magnitude
    penetration : float - insurance penetration rate (0 to 1)
    model : fitted LinearRegression
    xi_g, mu_g, sigma_g : GEV parameters fitted on residuals
    percentiles : list of floats

    Returns
    -------
    dict : {label: insured_loss_in_bn_usd}
    """
    X_new = np.array([[magnitude, np.log10(penetration)]])
    mu_pred = model.predict(X_new)[0]

    result = {}
    for p in percentiles:
        # GEV quantile: shift by mu_pred to get log10(loss) quantile
        log_loss_p = genextreme.ppf(p, xi_g, loc=mu_pred + mu_g, scale=sigma_g)
        result[f"p{int(p*100)}"] = round(10 ** log_loss_p, 4)
    return result


# Application: Japan M6.8 Kumamoto (28 July 2026)
# Penetration = 0.277 from EM-DAT (Japan, 16 observations, reliable)
JAPAN_PENETRATION_EMDAT = 0.277
loss_dist = predict_loss_distribution(6.8, JAPAN_PENETRATION_EMDAT, model, xi_g, mu_g, sigma_g)

print("LOSS DISTRIBUTION - Japan M6.8 Kumamoto (28 July 2026)")
print("=" * 55)
print(f"  Japan insurance penetration: {JAPAN_PENETRATION_EMDAT:.1%} (EM-DAT, 16 obs)")
print()
for k, v in loss_dist.items():
    print(f"  {k} : ${v:.4f}bn")
print()
print(f"Our p95 = ${loss_dist['p95']:.3f}bn")
print(f"Euler estimated $3-4.5bn - our p90 (${loss_dist['p90']:.3f}bn) is closest to this range")
print("The actual event was a high-percentile scenario - below most attachment points")

# Save all parameters for portfolio_impact.ipynb
model_params = {
    "intercept": float(model.intercept_),
    "coef_magnitude": float(model.coef_[0]),
    "coef_log_penetration": float(model.coef_[1]),
    "xi_g": float(xi_g),
    "mu_g": float(mu_g),
    "sigma_g": float(sigma_g),
    "distribution": "GEV on log10(loss) residuals",
    "r2": float(r2),
    "n_obs": len(df),
    "japan_penetration_emdat": JAPAN_PENETRATION_EMDAT
}
with open("model_params.json", "w") as f:
    json.dump(model_params, f, indent=2)
print("\nModel parameters saved to model_params.json")

LOSS DISTRIBUTION - Japan M6.8 Kumamoto (28 July 2026)
  Japan insurance penetration: 27.7% (EM-DAT, 16 obs)

  p10 : $0.0252bn
  p25 : $0.1029bn
  p50 : $0.4448bn
  p75 : $1.6303bn
  p90 : $4.2841bn
  p95 : $6.8183bn
  p99 : $12.7535bn

Our p95 = $6.818bn
Euler estimated $3-4.5bn - our p90 ($4.284bn) is closest to this range
The actual event was a high-percentile scenario - below most attachment points

Model parameters saved to model_params.json


## 7. Fan Chart - Loss Distribution by Magnitude

Key output: for each magnitude, the full uncertainty range of insured losses for Japan.
- Blue line = median prediction
- Dark band = IQR (p25-p75) - where 50% of scenarios fall
- Light band = p10-p90 - where 80% of scenarios fall
- Dotted green lines = ILS attachment points of simulated portfolio

Validation: Tohoku 2011 actual loss (~$35bn) should fall within the prediction band for M9.0.

In [18]:
# Japan penetration from EM-DAT (16 observations, reliable)
JAPAN_PENETRATION = 0.277

magnitudes = np.linspace(5.0, 9.5, 100)

def get_quantile_curve(magnitudes, p, penetration, model, xi, mu_gev, sigma_gev):
    """Compute loss quantile p for each magnitude value."""
    results = []
    for m in magnitudes:
        X_new = np.array([[m, np.log10(penetration)]])
        mu_pred = model.predict(X_new)[0]
        log_q = genextreme.ppf(p, xi, loc=mu_pred + mu_gev, scale=sigma_gev)
        results.append(10 ** log_q)
    return results

p10 = get_quantile_curve(magnitudes, 0.10, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)
p25 = get_quantile_curve(magnitudes, 0.25, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)
p50 = get_quantile_curve(magnitudes, 0.50, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)
p75 = get_quantile_curve(magnitudes, 0.75, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)
p90 = get_quantile_curve(magnitudes, 0.90, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)
p95 = get_quantile_curve(magnitudes, 0.95, JAPAN_PENETRATION, model, xi_g, mu_g, sigma_g)

fig = go.Figure()

# p10-p90 shaded band - 80% of scenarios
fig.add_trace(go.Scatter(
    x=list(magnitudes) + list(magnitudes[::-1]),
    y=p90 + p10[::-1],
    fill='toself', fillcolor='rgba(0,100,200,0.10)',
    line=dict(color='rgba(0,0,0,0)'),
    name='p10-p90 range (80%)'
))
# p25-p75 IQR band - 50% of scenarios
fig.add_trace(go.Scatter(
    x=list(magnitudes) + list(magnitudes[::-1]),
    y=p75 + p25[::-1],
    fill='toself', fillcolor='rgba(0,100,200,0.20)',
    line=dict(color='rgba(0,0,0,0)'),
    name='p25-p75 range IQR (50%)'
))
# p95 stress scenario
fig.add_trace(go.Scatter(
    x=magnitudes, y=p95,
    mode='lines', line=dict(color='red', dash='dot', width=1.5),
    name='p95 (stress scenario)'
))
# Median
fig.add_trace(go.Scatter(
    x=magnitudes, y=p50,
    mode='lines', line=dict(color='blue', width=2.5),
    name='Median (p50)'
))
# p10
fig.add_trace(go.Scatter(
    x=magnitudes, y=p10,
    mode='lines', line=dict(color='green', dash='dot', width=1),
    name='p10'
))
# Kumamoto M6.8 - actual event last week
fig.add_trace(go.Scatter(
    x=[6.8], y=[loss_dist["p50"]],
    mode='markers+text',
    text=[f"Kumamoto M6.8 (Jul 2026)<br>p50=${loss_dist['p50']}bn"],
    textposition="top right",
    marker=dict(size=12, color='red', symbol='star'),
    name='Japan M6.8 (Jul 2026)'
))
# Tohoku 2011 - external validation point
fig.add_trace(go.Scatter(
    x=[9.0], y=[35],
    mode='markers+text',
    text=["Tohoku 2011<br>$35bn (actual)"],
    textposition="top left",
    marker=dict(size=12, color='darkred', symbol='diamond'),
    name='Tohoku 2011 (actual - validation)'
))

fig.update_layout(
    title="Industry Insured Loss Distribution by Magnitude - Japan Earthquakes<br>"
          "<sub>Log-linear regression + GEV residuals | Source: EM-DAT 2000-2024 | "
          "n=76 events | Japan penetration=27.7% (EM-DAT)</sub>",
    xaxis_title="Magnitude",
    yaxis_title="Industry Insured Loss ($bn)",
    yaxis_type="log",
    legend=dict(x=0.01, y=0.99),
    height=550
)
fig.show(renderer="iframe")

# Validation checks
idx_9 = int((9.0 - 5.0) / (9.5 - 5.0) * 99)

print("Validation:")
print(f"  Kumamoto M6.8: p50=${loss_dist['p50']:.3f}bn | p90=${loss_dist['p90']:.3f}bn")
print(f"  Euler estimated $3-4.5bn -> closest to our p90 (${loss_dist['p90']:.3f}bn)")
print(f"  Median below $2bn -> consistent with no material ILS impact at median")
print()
print(f"  Tohoku 2011 (M9.0): actual $35bn")
print(f"  Model: p50=${p50[idx_9]:.1f}bn | p90=${p90[idx_9]:.1f}bn | p95=${p95[idx_9]:.1f}bn")
print(f"  Tohoku falls above p95 -> extreme tail event, model underestimates at M9.0+")
print(f"  Limitation: few M9+ events in 76-observation calibration dataset")

Validation:
  Kumamoto M6.8: p50=$0.445bn | p90=$4.284bn
  Euler estimated $3-4.5bn -> closest to our p90 ($4.284bn)
  Median below $2bn -> consistent with no material ILS impact at median

  Tohoku 2011 (M9.0): actual $35bn
  Model: p50=$1.8bn | p90=$17.7bn | p95=$28.2bn
  Tohoku falls above p95 -> extreme tail event, model underestimates at M9.0+
  Limitation: few M9+ events in 76-observation calibration dataset
